# Movie Data Analysis: Data Preparation & Cleaning

This notebook documents the data preparation and cleaning steps applied to multiple
movie-related datasets. The objective is to standardize structure, correct data types,
handle missing values, and prepare the data for analysis.

# Phase 1: Data Preparation
Prepares the raw data by loading, inspecting, and standardizing dataset structure.
It focuses on column naming, missing value assessment, and removal of unusable fields, ensuring the data is ready for further processing.

## Stage 0: Setup & Imports

In this stage, we import the required Python libraries and configure the Jupyter
environment to support data analysis and visualization.

In [ ]:
# Core data analysis libraries
import pandas as pd
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"  # to make jupyter print all outputs, not just the last one
from IPython.core.display import HTML  # to pretty print pandas df and be able to copy them over (e.g. to ppt slides)


## Stage 1: Load the Data

Each dataset represents a different aspect of movie data:
- ExpertReviews: Professional critic reviews and scores
- UserReviews: User-generated reviews and feedback
- Meta: Movie metadata such as genre, studio, and ratings
- Sales: Box office and financial performance data

In [ ]:
# Load datasets
# Each dataset represents a different aspect of movie data

expert_df = pd.read_csv("../Metacritic dataset/ExpertReviews.csv")

user_df = pd.read_csv(
    "../Metacritic dataset/UserReviews.csv",
    low_memory=False  # avoids dtype inference warnings
)

meta_df = pd.read_csv("../Metacritic dataset/metaClean43Brightspace.csv")

sales_df = pd.read_csv("../Metacritic dataset/sales.csv")


## Stage 2: Initial Data Inspection

We inspect the size and structure of each dataset by checking the number of rows,
columns, and column names.

In [ ]:
print("There are {} rows and {} columns in the ExpertReviews.csv ".format(expert_df.shape[0], expert_df.shape[1]))
print("There are {} rows and {} columns in the UserReviews.csv ".format(user_df.shape[0], user_df.shape[1]))
print("There are {} rows and {} columns in the metaClean43Brightspace.csv ".format(meta_df.shape[0], meta_df.shape[1]))
print("There are {} rows and {} columns in the sales.csv ".format(sales_df.shape[0], sales_df.shape[1]))

## Stage 3: Column Name Standardization

Column names are standardized to improve readability, consistency,
and ease of merging across datasets.

### 3.1 Inspect the column names

In [ ]:
print("Column names in ExpertReviews.csv:", list(expert_df.columns))
print("Columns names in UserReviews.csv ", list(user_df.columns))
print("Columns names in metaClean43Brightspace.csv ", list(meta_df.columns))
print("Columns names in sales.csv ", list(sales_df.columns))

### 3.2 Rename the column names

In [ ]:
expert_df = expert_df.rename(columns={
    'idvscore': 'individual_score',
    'dateP': 'publish_date',
    'Rev': 'review_text'
})

user_df = user_df.rename(columns={
    'idvscore': 'individual_score',
    'dateP': 'publish_date',
    'Rev': 'review_text',
    'thumbsUp': 'thumbs_up',
    'thumbsTot': 'thumbs_total'
})

meta_df = meta_df.rename(columns={
    'RelDate': 'release_date',
    'metascore': 'meta_score',
    'userscore': 'user_score'
})

sales_df = sales_df.rename(columns={
    'international_box_office': 'intl_box_office',
    'domestic_box_office': 'dom_box_office',
    'worldwide_box_office': 'global_box_office',
    'avg run per theatre': 'avg_run_per_theatre',
    'Unnamed: 8': 'opening_weekend_revenue'
})



### 3.3 Verify column name changes

In [ ]:
print("Column names in ExpertReviews.csv:", list(expert_df.columns))
print("Columns names in UserReviews.csv ", list(user_df.columns))
print("Columns names in metaClean43Brightspace.csv ", list(meta_df.columns))
print("Columns names in sales.csv ", list(sales_df.columns))

## Stage 4: Missing Value Analysis

We compute the number and percentage of missing values per column
to guide data cleaning decisions.

In [ ]:
def missing_summary(df, name):
    summary = (
        df.isna()
        .sum()
        .to_frame(name='Missing Values')
        .assign(Percentage=lambda x: (x['Missing Values'] / len(df) * 100).round(2))
    )
    summary = summary[summary['Missing Values'] > 0]

    print(f"\n{name} — Missing Values Summary")
    print(summary if not summary.empty else "No missing values 🎉")




In [ ]:
missing_summary(expert_df, "ExpertReviews")
missing_summary(user_df, "UserReviews")
missing_summary(meta_df, "Meta")
missing_summary(sales_df, "Sales")

## Stage 5: Removal of Unusable Columns

Columns that contain little or no meaningful information are removed
to simplify the datasets and reduce noise.

In [ ]:
# Drop fully empty columns
sales_df = sales_df.drop(columns=[
    'opening_weekend_revenue'
])

print("Columns names in sales.csv ", list(sales_df.columns))

# Phase 2: Data Type Correction and Semantic Cleaning

Phase 2 focuses on correcting and validating data types for each dataset individually.
The goal is to ensure semantic correctness and consistency before defining relationships
or performing any dataset integration.

## Stage 0: ExpertReviews

This stage focuses on inspecting and correcting data types in the ExpertReviews dataset.

During inspection, additional issues related to missing values and text formatting
were observed. Since these issues are limited to individual columns and do not
affect table relationships, basic normalization is performed as part of Phase 2.
More advanced text processing will be addressed in a later phase.

### 0.1 Data Type Inspection

We inspect both the data types and a sample of the data to understand
the structure and content of each column before applying any corrections.

In [ ]:
# Inspect data types
print(expert_df.dtypes)

# View sample rows
expert_df.head(50)


**Inspection summary:**

- `url` (object): Correct, used as an identifier
- `individual_score` (float64): Correct numeric type
- `reviewer` (object): Correct type, but contains missing values and inconsistent quoting
- `publish_date` (object): Incorrect type, should be datetime
- `review_text` (object): Correct type, but contains formatting artifacts and missing values

### 0.2 Data Type and Basic Semantic Corrections

Based on the inspection, we apply the following corrections:
- Convert `publish_date` to datetime
- Normalize missing values in text fields
- Remove obvious quoting artifacts from text columns

These corrections are limited to column-level normalization and do not
alter the semantic meaning of the data.
python
Copy code


In [ ]:
# Convert publish_date to datetime
expert_df['publish_date'] = pd.to_datetime(
    expert_df['publish_date'], errors='coerce'
)

# Normalize reviewer column
expert_df['reviewer'] = (
    expert_df['reviewer']
    .replace('None', pd.NA)
    .str.strip(' "\'')
)

# Normalize review_text column
expert_df['review_text'] = (
    expert_df['review_text']
    .astype(str)
    .str.strip(' "\'')
    .replace('None', pd.NA)
)


### 0.3 Post-correction Verification

We re-inspect a sample of the data to verify that the corrections
have been applied as intended.

In [ ]:
# View sample rows
expert_df.head(50)

## Stage 1: UserReviews

This stage focuses on inspecting and correcting data types and basic semantic issues
in the UserReviews dataset. Compared to ExpertReviews, this table contains additional
numeric counts and more frequent missing values.

### 1.1 Data Type Inspection

We inspect both the data types and a sample of the data to understand
the structure, content, and quality of each column before applying corrections.

In [ ]:
# Inspect data types
print(user_df.dtypes)

# Inspect sample rows
user_df.head(50)


**Inspection summary:**

- `url` (object): Correct, used as an identifier
- `individual_score` (object): Should be numeric
- `reviewer` (object): Correct type, contains missing values
- `publish_date` (object): Should be datetime
- `review_text` (object): Correct type, contains formatting artifacts and duplication
- `thumbs_up` (object): Should be integer count
- `thumbs_total` (object): Should be integer count


### 1.2 Data Type and Basic Semantic Corrections

Based on the inspection, we apply the following corrections:
- Convert numeric columns to appropriate numeric types
- Convert publish_date to datetime
- Normalize missing values in text columns
- Preserve text meaning while removing obvious artifacts


In [ ]:
# Drop rows that contain no usable review information
user_df = user_df.dropna(
    subset=['individual_score', 'review_text', 'publish_date'],
    how='all'
)

# Convert individual_score to numeric
user_df['individual_score'] = pd.to_numeric(
    user_df['individual_score'], errors='coerce'
)

# Clean and convert publish_date
user_df['publish_date'] = (
    user_df['publish_date']
    .astype(str)
    .str.strip(' "\'')
)

user_df['publish_date'] = pd.to_datetime(
    user_df['publish_date'], errors='coerce'
)

# Convert thumbs counts to nullable integers
user_df['thumbs_up'] = pd.to_numeric(
    user_df['thumbs_up'], errors='coerce'
).astype('Int64')

user_df['thumbs_total'] = pd.to_numeric(
    user_df['thumbs_total'], errors='coerce'
).astype('Int64')

# Normalize reviewer column
user_df['reviewer'] = (
    user_df['reviewer']
    .replace('None', pd.NA)
    .str.strip(' "\'')
)

# Normalize review_text column
user_df['review_text'] = (
    user_df['review_text']
    .astype(str)
    .str.strip(' "\'')
    .replace('None', pd.NA)
)



### 1.3 Post-correction Verification

We re-inspect a sample of the data to confirm that the corrections
have been applied correctly and that no unintended changes were introduced.


In [ ]:

# Verify sample rows
user_df.head(50)


## Stage 2: Meta

This stage focuses on inspecting and correcting data types and basic semantic issues
in the Meta dataset. This table contains descriptive movie metadata, including
categorical variables and numeric scores, which will later be used for grouping
and analysis.


### 2.1 Data Type Inspection

We inspect both the data types and a sample of the data to understand
the structure, content, and quality of each column before applying corrections.


In [ ]:
# Inspect data types
print(meta_df.dtypes)

# Inspect sample rows
meta_df.head(50)


**Inspection summary:**

Inspection of the Meta dataset shows that most text fields are semantically rich and
will be preserved for later analysis. In Phase 2, only structural corrections are
applied, including date parsing and categorical type assignment.

- `url` (object): Correct, used as an identifier
- `title` (object): Correct text field
- `studio` (object): Categorical variable, should be category
- `rating` (object): Categorical variable, should be category
- `runtime` (float64): Correct numeric type
- `cast` (object): Correct text field, contains missing values
- `director` (object): Correct text field, contains missing values
- `genre` (object): Categorical variable, should be category
- `summary` (object): Correct text field, contains missing values
- `awards` (object): Correct text field, contains many missing values
- `meta_score` (int64): Correct numeric type
- `user_score` (float64): Correct numeric type
- `release_date` (object): Should be datetime


### 2.2 Data Type and Basic Semantic Corrections

Based on the inspection, we apply the following corrections:
- Convert release_date to datetime
- Convert categorical variables to category type
- Preserve text fields as object without semantic alteration


In [ ]:
# Convert release_date to datetime
meta_df['release_date'] = pd.to_datetime(
    meta_df['release_date'], errors='coerce'
)

# Convert categorical columns to category
for col in ['studio', 'rating', 'genre']:
    meta_df[col] = meta_df[col].astype('category')


### 2.3 Post-correction Verification

We re-inspect a sample of the data to confirm that the corrections
have been applied correctly and that no unintended changes were introduced.


In [ ]:
# Verify sample rows
meta_df.head(50)


## Stage 3: Sales

This stage focuses on inspecting and correcting data types and basic structural issues
in the Sales dataset. This table contains financial and operational information and
exhibits a high proportion of missing values, requiring careful and conservative
cleaning decisions.


In [ ]:
# Inspect data types
print(sales_df.dtypes)

# Inspect sample rows
sales_df.head(50)


**Inspection summary:**

Inspection of the Sales dataset shows that the release_date column contains
inconsistent and non-standard values (e.g., partial dates and descriptive text).
To avoid introducing errors, the release_date column is preserved as text and
the year column is retained as the primary temporal reference in Phase 2.


- `year` (int64): Correct numeric type, primary temporal reference
- `release_date` (object / datetime64): Contains many missing or invalid values
- `title` (object): Correct text field
- `genre` (object): Categorical variable, should be category
- `intl_box_office` (float64): Numeric, contains many missing values
- `dom_box_office` (float64): Numeric, contains many missing values
- `global_box_office` (float64): Numeric, contains many missing values
- `production_budget` (float64): Numeric, high proportion of missing values
- `opening_weekend` (float64): Numeric, many missing values
- `theatre_count` (float64): Numeric, many missing values
- `avg_run_per_theatre` (float64): Numeric, many missing values
- `runtime` (float64): Numeric, partially missing
- `keywords` (object): Free-text field
- `creative_type` (object): Categorical variable, should be category
- `url` (object): Correct, used as an identifier



### 3.2 Data Type and Structural Corrections

Based on the inspection, we apply conservative corrections:
- Convert categorical variables to category type
- Retain numeric columns as floats to preserve missing values
- Preserve year as the primary temporal reference
- Avoid reconstructing release_date in Phase 2


In [ ]:
# Convert categorical columns
sales_df['genre'] = sales_df['genre'].astype('category')
sales_df['creative_type'] = sales_df['creative_type'].astype('category')


### 3.3 Post-correction Verification

We re-inspect the data types and a sample of the data to confirm that
the corrections have been applied correctly and that no unintended changes
were introduced.


In [ ]:
# Verify sample rows
sales_df.head(50)


# Phase 3: Relationships and Dataset Integration

Phase 3 focuses on identifying, validating, and defining relationships between
the datasets prepared in Phases 1 and 2. Before performing any merges, we analyze
key consistency, relationship cardinality, and potential sources of duplication
or data loss.

This phase ensures that dataset integration is performed deliberately and
correctly, rather than implicitly or through trial and error.


## Stage 0: Meta ↔ Sales Relationship Feasibility

The Sales dataset originates from a different source and does not share a
common URL structure with Meta. To evaluate whether a relationship can exist,
we perform a title-based intersection analysis without merging any data.

Meta is treated as the anchor table.


### 0.1 Title-based Intersection (NO MERGE)

Movie titles are normalized and compared to identify Meta movies that
also appear in the Sales dataset.


In [ ]:
# Normalize titles
meta_titles = (
    meta_df['title']
    .astype(str)
    .str.lower()
    .str.strip()
)

sales_titles = (
    sales_df['title']
    .astype(str)
    .str.lower()
    .str.strip()
)

# Create Sales title set
sales_title_set = set(sales_titles.dropna())

# Identify Meta titles that exist in Sales
meta_in_sales_mask = meta_titles.isin(sales_title_set)

# Summary
print("Meta movies with a Sales match:", meta_in_sales_mask.sum())
print(
    "Percentage of Meta movies covered by Sales:",
    f"{meta_in_sales_mask.mean() * 100:.2f}%"
)

# Inspect intersecting Meta rows
meta_df.loc[meta_in_sales_mask, ['title', 'release_date']].head(20)


## Stage 1: Meta ↔ ExpertReviews Relationship

The Meta and ExpertReviews datasets share a common URL identifier.
This stage validates coverage and confirms the expected one-to-many
relationship between movies and expert reviews.


### 1.1 URL-based Relationship Inspection (NO MERGE)


In [ ]:
meta_urls = set(meta_df['url'])
expert_urls = set(expert_df['url'])

# Intersection
common_urls = meta_urls.intersection(expert_urls)

print("Meta movies:", len(meta_urls))
print("Expert review movies:", len(expert_urls))
print("Common URLs:", len(common_urls))

print(
    "Percentage of Meta movies with ExpertReviews:",
    f"{len(common_urls) / len(meta_urls) * 100:.2f}%"
)

# Cardinality: number of expert reviews per movie
expert_review_counts = expert_df['url'].value_counts()

print("\nExpert review count summary:")
print(expert_review_counts.describe())


## Stage 2: Meta ↔ UserReviews Relationship

The Meta and UserReviews datasets also share a common URL identifier.
However, user reviews are optional, meaning not all movies receive
user-generated feedback. This stage validates coverage and review volume.


### 2.1 URL-based Relationship Inspection (NO MERGE)


In [ ]:
meta_urls = set(meta_df['url'])
user_urls = set(user_df['url'])

# Intersection
common_urls = meta_urls.intersection(user_urls)

print("Meta movies:", len(meta_urls))
print("User review movies:", len(user_urls))
print("Common URLs:", len(common_urls))

print(
    "Percentage of Meta movies with UserReviews:",
    f"{len(common_urls) / len(meta_urls) * 100:.2f}%"
)

# Cardinality: number of user reviews per movie
user_review_counts = user_df['url'].value_counts()

print("\nUser review count summary:")
print(user_review_counts.describe())


## Phase 3 Summary

- Meta serves as the anchor table, representing one row per movie.
- ExpertReviews and UserReviews exhibit clear one-to-many relationships
  with Meta using the URL identifier.
- The Sales dataset cannot be directly joined via URL but shows meaningful
  overlap when titles are used as a semantic linking key.
- No merges are performed in this phase; all relationships are evaluated
  conceptually and quantitatively to inform later integration decisions.


# Phase 4: Export EDA Artifacts

This final section of the EDA notebook exports the validated outputs and
design decisions produced during exploratory analysis. These artifacts are
intended to be consumed by downstream notebooks (e.g., feature engineering)
without re-running EDA logic.

The exported artifacts serve two purposes:
1. Persist cleaned, type-correct datasets
2. Persist relationship and integration decisions in a machine-readable form

This separation ensures reproducibility, prevents duplicated logic, and
enforces consistent integration rules across the project.


## Export 1: Cleaned and Validated Datasets

The datasets below reflect all cleaning, type correction, and validation
performed in Phases 1 and 2. These files should be treated as the canonical
inputs for feature engineering and modeling.


In [ ]:
# Save processed datasets
meta_df.to_csv("../cleaned_data/meta_clean.csv", index=False)
expert_df.to_csv("../cleaned_data/expert_reviews_clean.csv", index=False)
user_df.to_csv("../cleaned_data/user_reviews_clean.csv", index=False)
sales_df.to_csv("../cleaned_data/sales_clean.csv", index=False)


## Export 2: Relationship and Integration Decisions

During Phase 3, we analyzed potential relationships between datasets and
validated join feasibility, keys, and cardinality. Rather than encoding these
decisions implicitly in downstream code, we export them explicitly as a
configuration artifact.

This artifact defines:
- The anchor table for integration
- Safe one-to-many joins
- Probabilistic or unsafe joins that require caution
- Notes explaining integration constraints

Downstream notebooks are expected to *read* this artifact and respect its rules.


## Completion Summary

At the conclusion of this notebook:

- All datasets have been cleaned and type-corrected
- Meta has been established as the anchor entity
- Deterministic one-to-many relationships with ExpertReviews and UserReviews
  have been validated
- Sales integration has been identified as probabilistic and deferred
- All decisions have been exported as explicit artifacts

Subsequent notebooks (e.g., feature engineering) should rely exclusively on
the exported datasets and configuration files, and should not re-run or
duplicate EDA logic.
